In [2]:
import json
import os
from datetime import timedelta
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (mean_squared_error,
                             r2_score, mean_absolute_error,
                             mean_absolute_percentage_error)
from numpy import sqrt



In [3]:
def error_by_horizon(df_pred, df_ref):
    """
    Calculate error metrics for each horizon in the prediction DataFrame.
    """
    horizons = df_pred.columns
    error_metrics = {}

    for horizon in horizons:
        mse = mean_squared_error(df_ref[horizon], df_pred[horizon])
        r2 = r2_score(df_ref[horizon], df_pred[horizon])
        mae = mean_absolute_error(df_ref[horizon], df_pred[horizon])
        mape = mean_absolute_percentage_error(df_ref[horizon], df_pred[horizon])
        rmse = sqrt(mse)

        error_metrics[horizon] = {
            'MSE': mse,
            'R2': r2,
            'MAE': mae,
            'MAPE': mape,
            'RMSE': rmse
        }

    return pd.DataFrame(error_metrics).T

def general_error(df_pred, df_ref):
    """
    Calculate general error metrics for the entire prediction DataFrame.
    """
    mse = mean_squared_error(df_ref, df_pred)
    r2 = r2_score(df_ref, df_pred)
    mae = mean_absolute_error(df_ref, df_pred)
    mape = mean_absolute_percentage_error(df_ref, df_pred)
    rmse = float(sqrt(mse))

    return {
        'MSE': mse,
        'R2': r2,
        'MAE': mae,
        'MAPE': mape,
        'RMSE': rmse
    }

def general_error_by_file(df_pred_path, df_ref_path):
    """
    Load prediction and reference DataFrames from CSV files and calculate general error metrics.
    """
    df_pred = pd.read_csv(df_pred_path, index_col=0, parse_dates=True)
    df_ref = pd.read_csv(df_ref_path, index_col=0, parse_dates=True)
    #df_pred.drop(columns=['extra'], inplace=True)

    return general_error(df_pred, df_ref)

def error_by_horizon_by_file(exp_path):
    """
    Load prediction and reference DataFrames from CSV files and calculate error metrics.
    """

    df_pred = pd.read_csv(exp_path+"/pred.csv", index_col=0, parse_dates=True)
    df_ref = pd.read_csv(exp_path + "/ref.csv", index_col=0, parse_dates=True)
    #df_pred.drop(columns=['extra'], inplace=True)

    return error_by_horizon(df_pred, df_ref)

def traspose_series_from_df_row(df_pred, row):
    aux_df = df_pred.reset_index()
    transposed_values = aux_df.iloc[row,:].values
    #transposed_values.extend(aux_df.iloc[:,1].values)
    dates = [transposed_values[0] + timedelta(weeks=i) for i in range(len(transposed_values) -1)]
    transposed_series = pd.Series(transposed_values[1:], index=dates, name='val')
    return transposed_series

def plot_all_ref_pred(df_pred, df_ref):

    for i in range(8):
        traspose_series_pred = traspose_series_from_df_row(df_pred, i)
        traspose_series_ref = traspose_series_from_df_row(df_ref, i)

        #plot

        plt.figure(figsize=(10, 5))
        traspose_series_pred.plot(label='Prediction')
        traspose_series_ref.plot(label='Reference')
        plt.title('Prediction vs Reference')
        plt.xlabel('Date')
        plt.ylabel('Value')
        plt.legend()
        plt.grid()
        plt.show()

In [4]:

def create_comparison_df(experiment_dir):
    folders = []
    for entry in os.listdir(experiment_dir):
        full_path = os.path.join(experiment_dir, entry)
        if os.path.isdir(full_path):
            folders.append(os.path.abspath(full_path))

    error_list = [
    general_error_by_file(f + "/pred.csv", f + "/ref.csv")
    for f in folders]


    configs = []
    for f in folders:
        with open(f+"/config.json", "r") as file:
            configs.append(json.load(file))

    configs_df = pd.DataFrame(configs)

    error_df = pd.DataFrame(error_list)
    error_df["id"] = index = [f.split("/")[-1] for f in folders]
    result_df = pd.concat([configs_df['model'], error_df], axis="columns")
    result_df.set_index("id", inplace=True)

    return result_df

error_df = create_comparison_df("../../experiments/exp_2025-07-05")
error_df

,model,MSE,R2,MAE,MAPE,RMSE
id,,,,,,
res_config4.json,sarimax,0.058133,0.345127,0.189460,0.327292,0.241107
res_config7.json,simple_lstm_v0,0.118335,-0.299306,0.268191,0.495190,0.343998
res_config1.json,prophet,0.090322,0.085595,0.251205,0.454444,0.300536
res_config7.2.json,simple_lstm_v0,0.083422,0.139424,0.221317,0.314496,0.288829
res_config3.json,simple_lstm_v0,0.159800,-0.641945,0.325604,0.549089,0.399750
res_config5.2.json,sarimax,0.049174,0.464411,0.179611,0.290902,0.221752
res_config2.json,sarimax,0.120937,-0.421304,0.266777,0.428928,0.347760
res_config1.2.json,prophet,0.091939,0.068852,0.254987,0.469953,0.303215
res_config3.2.json,simple_lstm_v0,0.092712,0.054702,0.241504,0.383161,0.304486


---


In [138]:
error_by_horizon_by_file("../../experiments/exp_2025-07-05/res_config1.json")

,MSE,R2,MAE,MAPE,RMSE
1,0.101352,0.178355,0.273831,0.492004,0.318358
2,0.141539,0.078572,0.291898,0.427137,0.376216
3,0.094355,0.190444,0.255935,0.451853,0.307172
4,0.063573,0.243069,0.195601,0.387453,0.252136
5,0.065929,0.051926,0.223718,0.411673,0.256766
6,0.064697,-0.016511,0.219183,0.431192,0.254357
7,0.080798,-0.029577,0.257458,0.498784,0.284250
8,0.110335,-0.011517,0.292015,0.535451,0.332168


In [139]:
error_by_horizon_by_file("../../experiments/exp_2025-07-05/res_config3.json")

,MSE,R2,MAE,MAPE,RMSE
1,0.133813,-0.084804,0.261698,0.389801,0.365804
2,0.242095,-0.576057,0.357334,0.512493,0.492031
3,0.186686,-0.601749,0.347291,0.550238,0.432071
4,0.142903,-0.701481,0.310758,0.542887,0.378025
5,0.128380,-0.846150,0.300615,0.532295,0.358302
6,0.106546,-0.674028,0.296826,0.544707,0.326414
7,0.154636,-0.970461,0.345000,0.621628,0.393237
8,0.183343,-0.680828,0.385308,0.698662,0.428186
